In [ ]:
import json
json_path = "file_dicts.json"
with open(json_path, "r") as f:
    file_dicts = json.load(f)
file_dicts[0]
import nibabel as nib 
for v in file_dicts[0].values():
    if v.endswith(".nii.gz"):
        img = nib.load(v).get_fdata()
        print(img.shape)
file_dict_one = []
for file_dict in file_dicts:
    file_dict_one.append({"image":file_dict["image"]})
    file_dict_one.append({"image":file_dict["source_image"]})
print(len(file_dict_one))

In [1]:
import monai 
from monai.bundle.config_parser import ConfigParser
import torch 
# !mkdir models
# !cp  /data1/syliu/miccai24_maisi_SR/models/autoencoder_epoch273.pt models/autoencoder_epoch273.pt 
config= ConfigParser()
config.read_config("shiyu_utils/config_maisi3d-rflow.json")
autoencoder = config.get_parsed_content("autoencoder_def")
autoencoder.load_state_dict(torch.load("models/autoencoder_epoch273.pt"))

/data/home/syliu/.conda/envs/liushiyu/lib/python3.12/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/data/home/syliu/.conda/envs/liushiyu/lib/python3.12/site-packages/torch/cuda/__init__.py:843: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


[2026-01-31 22:10:17,974] [WARNING] [real_accelerator.py:209:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.


/data/home/syliu/.conda/envs/liushiyu/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/data/home/syliu/.conda/envs/liushiyu/lib/python3.12/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/data/home/syliu/.conda/envs/liushiyu/lib/python3.12/site-packages/torch/cuda/__init__.py:843: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count
/tmp/ipykernel_2932253/3119307258.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to cons

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
from shiyu_utils.maisi_transforms import VAE_Transform
transform = VAE_Transform(
    is_train=False,
    random_aug=False,
    val_patch_size=(160,256,256),
    output_dtype=torch.float32,
    spacing_type="original",
    image_keys=["image"]
)
transform = transform.transform_dict["ct"]
transform.transforms[-3].scaler.a_min=-10
transform.transforms[-3].scaler.a_max=0
transform

In [ ]:
!ls {ixi_mcx_2025_latent}

In [5]:
import monai.data as md 
dataset = md.Dataset(data=file_dict_one,transform=transform)
dataloader = md.DataLoader(dataset,batch_size=1,shuffle=False,num_workers=8)
save_data_root ="ixi_mcx_2025_latent"
import os 
import torch 
import shutil 
os.makedirs(save_data_root,exist_ok=True)
autoencoder = autoencoder.cuda().eval()
from tqdm import tqdm
for batch in tqdm(dataloader):
    file_name=batch["image"].meta["filename_or_obj"][0].replace(".nii","").replace(".gz","")
    base_name = os.path.basename(file_name)
    label_name = os.path.basename(os.path.dirname(file_name))
    subject_name = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(file_name))))
    save_path = os.path.join(save_data_root,subject_name+"_"+base_name+"_"+label_name+".pt")
    if os.path.exists(save_path):
        continue
    with torch.no_grad(),torch.cuda.amp.autocast(True):
        latent = autoencoder.encode_stage_2_inputs(batch['image'].float().cuda())
        latent = latent.cpu().detach()
        torch.save(latent,save_path.replace(".nii.gz",".pt"))
        print("saving latent:",save_path)


 46%|████▌     | 9542/20656 [3:28:05<4:02:07,  1.31s/it]

saving latent: ixi_mcx_2025_latent/m2m_IXI275-HH-_MCX_FC1_p250_t8_results_log_simple.pt


 46%|████▌     | 9543/20656 [3:28:07<4:01:29,  1.30s/it]

saving latent: ixi_mcx_2025_latent/m2m_IXI275-HH-_MCX_T10_p250_t8_results_log_full.pt


 46%|████▌     | 9544/20656 [3:28:08<4:01:43,  1.31s/it]

...

saving latent: ixi_mcx_2025_latent/m2m_IXI338-HH-_MCX_FC6_p250_t8_results_log_simple.pt


 55%|█████▌    | 11363/20656 [4:09:08<5:08:00,  1.99s/it]

saving latent: ixi_mcx_2025_latent/m2m_IXI338-HH-_MCX_TP7_p250_t8_results_log_full.pt


 55%|█████▌    | 11363/20656 [4:09:10<3:23:46,  1.32s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.25 GiB. GPU 0 has a total capacity of 23.69 GiB of which 1.18 GiB is free. Including non-PyTorch memory, this process has 4.27 GiB memory in use. Process 3178926 has 2.21 GiB memory in use. Process 3617073 has 15.99 GiB memory in use. Of the allocated memory 3.95 GiB is allocated by PyTorch, and 25.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
batch = next(iter(dataloader))

In [ ]:
batch["image"].meta["filename_or_obj"][0]